# perturbaudit quick start: audit a network against a real Perturb-seq screen

This tutorial uses the Papalexi et al. 2021 ECCITE-seq screen (THP-1 cells, Cas9 knockout of 25 genes, 3–4 guides per gene), downloaded from scPerturb (Zenodo 7041849, 147 MB, CC BY 4.0). It runs in a few minutes on a laptop.

Steps: load the screen, build a `Screen`, infer a simple network, audit it, and look at the verdicts.

In [1]:
import os, re, urllib.request, numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp
from perturbaudit.screen import Screen
from perturbaudit.api import audit_network
from perturbaudit.report import html_report

f = "PapalexiSatija2021_eccite_RNA.h5ad"
if not os.path.exists(f):
    urllib.request.urlretrieve(f"https://zenodo.org/records/7041849/files/{f}?download=1", f)
A = ad.read_h5ad(f)
A.shape

(20729, 18649)

Labels in this file are per guide (`STAT1g2`). The targeted gene is the label without the guide suffix; controls are labelled `control`. Counts are normalised to CP10k + log1p.

In [2]:
guide = A.obs["perturbation"].astype(str).values
gene = np.array(["non-targeting" if g == "control" else re.sub(r"g\d+$", "", g) for g in guide])
X = A.X.tocsr() if sp.issparse(A.X) else sp.csr_matrix(A.X)
tot = np.asarray(X.sum(1)).ravel(); X = sp.diags(1e4 / np.maximum(tot, 1)) @ X; X.data = np.log1p(X.data)
genes = [g for g in sorted(set(gene) - {"non-targeting"}) if g in set(A.var_names)]
print(len(genes), "perturbed genes measured:", genes)

25 perturbed genes measured: [np.str_('ATF2'), np.str_('BRD4'), np.str_('CAV1'), np.str_('CD86'), np.str_('CMTM6'), np.str_('CUL3'), np.str_('ETV7'), np.str_('IFNGR1'), np.str_('IFNGR2'), np.str_('IRF1'), np.str_('IRF7'), np.str_('JAK2'), np.str_('MARCH8'), np.str_('MYC'), np.str_('NFKBIA'), np.str_('PDCD1LG2'), np.str_('POU2F2'), np.str_('SMAD4'), np.str_('SPI1'), np.str_('STAT1'), np.str_('STAT2'), np.str_('STAT3'), np.str_('STAT5A'), np.str_('TNFRSF14'), np.str_('UBE2L6')]


Build the screen on the perturbed genes (the parents and children of the network must be measured). Guide identity enables the independent-guide check.

In [3]:
cols = [list(A.var_names).index(g) for g in genes]
screen = Screen(X[:, cols], gene, genes, control_label="non-targeting", guide=guide)
screen.knockdown().round(1).sort_values().head(10)

STAT1      -59.4
JAK2       -32.1
IFNGR2     -13.0
STAT2      -12.1
UBE2L6     -12.0
CMTM6      -11.3
BRD4        -8.0
STAT3       -7.9
IRF1        -7.7
TNFRSF14    -7.6
dtype: float64

Any network can be audited: an edge list (`parent, child, score`) or a square score matrix. Here, a simple correlation network among control cells — the kind of edge set an observational method would produce.

In [4]:
ctrl = gene == "non-targeting"
C = np.corrcoef(X[ctrl][:, cols].toarray().T); np.fill_diagonal(C, 0); C = np.nan_to_num(np.abs(C))
edges, summary = audit_network(screen, C, genes, top_k=100)
summary

,top_k,n_testable,audited_fraction_of_testable,untestable_parent_not_perturbed,untestable_child_not_measured,untestable_ineffective_knockdown,reversed,unsupported,offtarget,neighbour,guide_specific,unstable,audited
0,25,24,0.292,0,0,1,3,13,0,0,1,0,7
1,50,48,0.292,0,0,2,7,25,0,0,2,0,14
2,100,91,0.352,0,0,9,16,41,0,0,2,0,32


In [5]:
edges[["rank", "parent", "child", "verdict", "z_parent_kd_on_child", "z_child_kd_on_parent", "guide_z"]].head(15)

,rank,parent,child,verdict,z_parent_kd_on_child,z_child_kd_on_parent,guide_z
0,1,SPI1,IFNGR2,unsupported,2.613178,-1.454978,0.5|2.8
1,2,IFNGR2,SPI1,unsupported,-1.454978,2.613178,-2.4|-0.6
2,3,IRF1,IFNGR2,reversed,-1.895168,-57.009422,-0.5|-1.5
3,4,IFNGR2,IRF1,audited,-57.009422,-1.895168,-49.6|-33.1
4,5,NFKBIA,IRF7,unsupported,0.840359,-1.094056,1.8|-1.1
5,6,IRF7,NFKBIA,unsupported,-1.094056,0.840359,-0.5|-1.5
6,7,NFKBIA,IFNGR2,reversed,-0.737668,9.143541,-0.4|-0.6
7,8,IFNGR2,NFKBIA,audited,9.143541,-0.737668,7.3|3.8
8,9,IRF1,JAK2,audited,-18.794063,-58.069468,-19.5|-4.9
9,10,JAK2,IRF1,audited,-58.069468,-18.794063,-39.5|-29.9


Write a self-contained HTML report (open it in any browser):

In [6]:
open("papalexi_report.html", "w", encoding="utf-8").write(html_report(edges, summary, title="Papalexi 2021: correlation network audit"))
print("wrote papalexi_report.html")

wrote papalexi_report.html


### Reading the verdicts
* **audited** – the parent's knockdown moves the child and no other check fails.
* **unsupported** – the screen's own knockdown of the parent does not move the child.
* **reversed** – unsupported, and the child's knockdown strongly moves the parent instead.
* **guide_specific** – one guide moves the child, another does not (possible off-target or efficiency effect).
* **untestable_*** – the parent was not perturbed, the child not measured, or the knockdown was ineffective.

The same call works from the command line: `perturbaudit audit --h5ad screen.h5ad --network edges.csv --perturbation-col gene --guide-col guide --html --out results/run1`.